# IOv52-HST — Merge Base + Trained Horizon Predictor

**Strategy:**
1. Build full **gate05** architecture (`IOHSTCombined` with `ChunkDecoderWithCache`, `CHUNK_SIZE=32`, `MAX_SEQ_LEN=256`)
2. Load `checkpoint_step_5000` → full base weights
3. **Swap** `model.horizon_predictor` with gate15's `HarmonicHorizonPredictor` (cross-attn/causal-attn version)
4. Load `merged_step_900` → overlay trained HP weights only (`strict=False`)
5. Run inference — generation via `chunk_decoder` logits, horizon inspection via trained HP

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--no-warn-conflicts", *pkgs], check=True)

pip(
    "transformers>=4.40.0",
    "huggingface_hub>=0.23.0",
    "accelerate>=0.30.0",
    "bitsandbytes>=0.43.0",
)
print('✅ Dependencies ready')

In [ ]:
# ── Cell 2: HF Token ─────────────────────────────────────────
import os

HF_TOKEN = ""  # ← paste here if not using Colab Secrets

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
        if HF_TOKEN:
            print("✅ HF token loaded from Colab Secrets")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    raise RuntimeError("⛔ No HF token! Set HF_TOKEN in Colab Secrets or paste above.")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"✅ Token set ({HF_TOKEN[:4]}...{HF_TOKEN[-4:]})")

In [ ]:
# ── Cell 3: Imports & GPU check ───────────────────────────────
import gc, math, time, warnings
from typing import Dict, Tuple, Optional, List, Any

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} "
              f"— {free/1e9:.1f} / {total/1e9:.1f} GB free")

In [ ]:
# ── Cell 4: gate05 Config (base model config) ─────────────────
class Config:
    D_MODEL: int            = 1280
    N_HEADS: int            = 10
    N_LAYERS: int           = 24
    N_TOP_LAYERS: int       = 4
    N_CHUNK_ENC_LAYERS: int = 2
    N_CHUNK_DEC_LAYERS: int = 2
    N_LATTICE_LAYERS: int   = 3
    MAX_SEQ_LEN: int        = 256   # gate05 value
    VOCAB_SIZE: int         = 50257
    CHUNK_SIZE: int         = 32    # gate05 value
    HORIZON: int            = 8
    N_MAMBA_LAYERS: int     = 3
    SSM_EXPAND: int         = 1
    SSM_D_STATE: int        = 16
    CIF_THRESHOLD: float    = 0.5

    GEN_TEMPERATURE: float  = 0.85
    TOP_P: float            = 0.9
    REPETITION_PENALTY: float = 1.3

    DEVICE = DEVICE

print("✅ Config defined")
print(f"   MAX_SEQ_LEN={Config.MAX_SEQ_LEN}  CHUNK_SIZE={Config.CHUNK_SIZE}  "
      f"D_MODEL={Config.D_MODEL}  N_LAYERS={Config.N_LAYERS}")

In [ ]:
# ── Cell 5: Full gate05 Architecture ─────────────────────────
# Every module exactly as in gate05.py

# ── SelectiveSSM ─────────────────────────────────────────────
class SelectiveSSM(nn.Module):
    def __init__(self, d_model: int, d_state: int = 8, d_conv: int = 4, expand: int = 1):
        super().__init__()
        self.d_model  = d_model
        self.d_inner  = d_model * expand
        self.d_state  = d_state
        self.in_proj  = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv     = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                   padding=d_conv - 1, groups=self.d_inner, bias=True)
        self.x_proj   = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj  = nn.Linear(1, self.d_inner, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(self.d_inner, -1)
        self.log_A    = nn.Parameter(torch.log(A))
        self.D        = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm     = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        B, L, D = x.shape
        xz            = self.in_proj(x)
        x_main, z     = xz.chunk(2, dim=-1)
        x_conv        = self.conv(x_main.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_act         = F.silu(x_conv)
        ssm_p         = self.x_proj(x_act)
        B_mat, C_mat, dt_raw = ssm_p.split([self.d_state, self.d_state, 1], dim=-1)
        dt            = F.softplus(self.dt_proj(dt_raw))
        A             = -torch.exp(self.log_A)
        dA            = torch.exp(dt.unsqueeze(-1) * A)
        dB            = dt.unsqueeze(-1) * B_mat.unsqueeze(2)
        cum_A         = torch.exp(torch.cumsum(torch.log(dA.clamp(min=1e-8)), dim=1))
        contrib       = dB * x_act.unsqueeze(-1)
        h_approx      = torch.cumsum(contrib / cum_A.clamp(min=1e-8), dim=1) * cum_A
        y             = (h_approx * C_mat.unsqueeze(2)).sum(-1)
        out           = y + self.D * x_act
        out           = out * F.silu(z)
        return self.norm(self.out_proj(out) + residual)


# ── DiamondMixer ─────────────────────────────────────────────
class DiamondMixer(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.split_proj = nn.Linear(d_model, d_model * 2)
        self.z_net = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Linear(d_model * 4, d_model))
        self.w_net = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Linear(d_model * 4, d_model))
        self.merge_proj = nn.Linear(d_model * 2, d_model)
        self.norm       = nn.LayerNorm(d_model)

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        xy   = self.split_proj(u)
        x, y = xy.chunk(2, dim=-1)
        z    = self.z_net(x + y)
        w    = self.w_net(y - x)
        return self.norm(u + self.merge_proj(torch.cat([z, w], dim=-1)))


# ── HebbianFastWeights ────────────────────────────────────────
class HebbianFastWeights(nn.Module):
    def __init__(self, d_model: int, lambda_decay: float = 0.95):
        super().__init__()
        self.lambda_decay = lambda_decay
        self.qkv  = nn.Linear(d_model, d_model * 3, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, D = x.shape
        qkv     = self.qkv(x).reshape(B, S, 3, D).permute(2, 0, 1, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        kv      = torch.einsum('bsd,bse->bde', k, v) * self.lambda_decay
        out     = torch.einsum('bsd,bde->bse', q, kv)
        lr      = torch.sigmoid((q * k).sum(dim=-1, keepdim=True))
        return self.norm(x + out * lr)


# ── CIFModule ────────────────────────────────────────────────
class CIFModule(nn.Module):
    def __init__(self, d_model: int, threshold: float = 0.5):
        super().__init__()
        self.chunk_size  = max(1, round(1.0 / threshold))
        self.weight_proj = nn.Linear(d_model, 1)
        nn.init.constant_(self.weight_proj.bias, -4.85)
        nn.init.zeros_(self.weight_proj.weight)
        self.value_proj  = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, L, D  = x.shape
        cs       = self.chunk_size
        alphas   = torch.sigmoid(self.weight_proj(x)).squeeze(-1)
        values   = self.value_proj(x)
        pad      = (cs - L % cs) % cs
        if pad:
            values   = F.pad(values,  (0, 0, 0, pad))
            alphas_p = F.pad(alphas,  (0, pad))
        else:
            alphas_p = alphas
        n_chunks = (L + pad) // cs
        v_chunks = values.reshape(B, n_chunks, cs, D)
        a_chunks = alphas_p.reshape(B, n_chunks, cs).unsqueeze(-1)
        denom    = a_chunks.sum(2).clamp(min=1e-3)
        fired    = (v_chunks * a_chunks).sum(2) / denom
        return fired, alphas


# ── SelfAttentionWithCache ────────────────────────────────────
class SelfAttentionWithCache(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.qkv      = nn.Linear(d_model, d_model * 3, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor,
                layer_past: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
                ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        B, S, D = x.shape
        qkv     = self.qkv(x).reshape(B, S, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        if layer_past is not None:
            k = torch.cat([layer_past[0], k], dim=2)
            v = torch.cat([layer_past[1], v], dim=2)
        present   = (k, v)
        is_causal = (layer_past is None)
        attn_out  = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
        out       = attn_out.transpose(1, 2).contiguous().view(B, S, D)
        return self.out_proj(out), present


# ── TransformerBlock ─────────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int,
                 use_diamond_ffn: bool = False, dropout: float = 0.1):
        super().__init__()
        self.attn        = SelfAttentionWithCache(d_model, n_heads)
        self.norm1       = nn.LayerNorm(d_model)
        self.norm2       = nn.LayerNorm(d_model)
        self.use_diamond = use_diamond_ffn
        self.drop        = nn.Dropout(dropout)
        if use_diamond_ffn:
            self.ffn: nn.Module = DiamondMixer(d_model)
        else:
            self.ff1 = nn.Linear(d_model, 4 * d_model)
            self.ff2 = nn.Linear(4 * d_model, d_model)
            self.act = nn.GELU(approximate="tanh")

    def forward(self, x: torch.Tensor,
                layer_past: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
                ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        attn_out, present = self.attn(self.norm1(x), layer_past)
        x = x + self.drop(attn_out)
        if self.use_diamond:
            x = self.ffn(x)
        else:
            x = x + self.drop(self.ff2(self.act(self.ff1(self.norm2(x)))))
        return x, present


# ── AdaptiveBlock ─────────────────────────────────────────────
class AdaptiveBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int,
                 use_ssm: bool = False, use_hebbian: bool = False):
        super().__init__()
        self.block       = TransformerBlock(d_model, n_heads, use_diamond_ffn=False)
        self.use_ssm     = use_ssm
        self.use_hebbian = use_hebbian
        if use_ssm:
            self.ssm     = SelectiveSSM(d_model, d_state=Config.SSM_D_STATE,
                                        expand=Config.SSM_EXPAND)
        if use_hebbian:
            self.hebbian = HebbianFastWeights(d_model)
        self.confidence_head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(d_model, 1), nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor,
                layer_past: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
                ) -> Tuple[torch.Tensor, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        x_out, present = self.block(x, layer_past)
        if self.use_ssm:
            x_out = self.ssm(x_out)
        if self.use_hebbian:
            x_out = self.hebbian(x_out)
        conf = (
            self.confidence_head(x_out.transpose(1, 2)).mean(0)
            if x_out.size(1) > 1
            else x_out.new_tensor([0.0])
        )
        return x_out, conf, present


# ── DiamondCrossAttention ─────────────────────────────────────
class DiamondCrossAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.q        = nn.Linear(d_model, d_model, bias=False)
        self.k        = nn.Linear(d_model, d_model, bias=False)
        self.v        = nn.Linear(d_model, d_model, bias=False)
        self.out      = nn.Linear(d_model, d_model, bias=False)
        self.norm     = nn.LayerNorm(d_model)
        self.gate     = nn.Parameter(torch.zeros(1))

    def forward(self, top_h: torch.Tensor, bottom_h: torch.Tensor) -> torch.Tensor:
        B, S, D = top_h.shape
        q = self.q(top_h).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k(bottom_h).view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v(bottom_h).view(B, -1, self.n_heads, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        out = attn_out.transpose(1, 2).contiguous().view(B, S, D)
        return self.norm(top_h + torch.tanh(self.gate) * self.out(out))


# ── ChunkEncoder ─────────────────────────────────────────────
class ChunkEncoder(nn.Module):
    def __init__(self, d_model: int, chunk_size: int = 128,
                 n_heads: int = 8, n_layers: int = 2):
        super().__init__()
        self.chunk_size    = chunk_size
        enc_layer          = nn.TransformerEncoderLayer(
            d_model, n_heads, d_model * 4, batch_first=True, dropout=0.0)
        self.local_encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.pooling_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pooling_attn  = nn.MultiheadAttention(d_model, n_heads, batch_first=True)

    def forward(self, token_embeddings: torch.Tensor) -> torch.Tensor:
        B, total, D = token_embeddings.shape
        cs          = self.chunk_size
        n_chunks    = total // cs
        chunks      = token_embeddings[:, :n_chunks * cs, :].view(B * n_chunks, cs, D)
        encoded     = self.local_encoder(chunks)
        query       = self.pooling_query.expand(B * n_chunks, -1, -1)
        pooled, _   = self.pooling_attn(query, encoded, encoded)
        return pooled.view(B, n_chunks, D)


# ── TransformerDecoderLayerWithCache ─────────────────────────
class TransformerDecoderLayerWithCache(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        ff_dim         = 4 * d_model
        self.self_attn = SelfAttentionWithCache(d_model, n_heads)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.linear1   = nn.Linear(d_model, ff_dim)
        self.linear2   = nn.Linear(ff_dim, d_model)
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.norm3     = nn.LayerNorm(d_model)
        self.drop      = nn.Dropout(dropout)

    def forward(self, tgt, memory, self_attn_past=None, cross_attn_past=None):
        sa_out, sa_present = self.self_attn(self.norm1(tgt), layer_past=self_attn_past)
        tgt = tgt + self.drop(sa_out)
        if cross_attn_past is not None:
            ca_out, _ = self.cross_attn(self.norm2(tgt),
                                        cross_attn_past[0], cross_attn_past[1])
            ca_present = cross_attn_past
        else:
            ca_out, _ = self.cross_attn(self.norm2(tgt), memory, memory)
            ca_present = (memory, memory)
        tgt = tgt + self.drop(ca_out)
        ff_out = self.linear2(self.drop(F.gelu(self.linear1(self.norm3(tgt)))))
        tgt = tgt + self.drop(ff_out)
        return tgt, sa_present, ca_present


# ── ChunkDecoderWithCache ─────────────────────────────────────
class ChunkDecoderWithCache(nn.Module):
    def __init__(self, d_model: int, vocab_size: int,
                 chunk_size: int = 128, n_heads: int = 8, n_layers: int = 2):
        super().__init__()
        self.chunk_size    = chunk_size
        self.pos_embedding = nn.Embedding(chunk_size, d_model)
        self.layers        = nn.ModuleList([
            TransformerDecoderLayerWithCache(d_model, n_heads)
            for _ in range(n_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, chunk_embeddings: torch.Tensor,
                target_token_embeddings: torch.Tensor,
                cache=None) -> Tuple[torch.Tensor, list]:
        B, S, D  = target_token_embeddings.shape
        device   = target_token_embeddings.device
        past_len = cache[0][0][0].size(2) if cache else 0
        positions = torch.arange(past_len, past_len + S,
                                 dtype=torch.long, device=device) % self.chunk_size
        tgt       = target_token_embeddings + self.pos_embedding(positions)
        new_cache = []
        for i, layer in enumerate(self.layers):
            layer_cache      = cache[i] if cache else (None, None)
            sa_past, ca_past = layer_cache
            chunk_idx        = min(
                (past_len // self.chunk_size),
                chunk_embeddings.size(1) - 1
            )
            memory = chunk_embeddings[:, chunk_idx:chunk_idx+1, :].expand(B, S, D)
            tgt, sa_present, ca_present = layer(tgt, memory, sa_past, ca_past)
            new_cache.append((sa_present, ca_present))
        logits = self.lm_head(tgt)
        return logits, new_cache


# ── FullLatticeFieldAnalyzer ──────────────────────────────────
class FullLatticeFieldAnalyzer(nn.Module):
    def __init__(self, max_seq_len: int = 8192):
        super().__init__()
        spine = [0, 2, 4]
        while True:
            nv = 2 * spine[-1] + 2 * spine[-2] + 2 * spine[-3]
            if nv >= max_seq_len:
                break
            spine.append(nv)
        self.register_buffer('spine', torch.tensor(spine, dtype=torch.long))
        self.max_depth         = len(spine)
        self.lattice_structure = {}
        for pos in spine:
            if pos < max_seq_len:
                self.lattice_structure[pos] = self._analyze_position(pos)
        self._non_spine_cache: Dict[int, Any] = {}

    def get_structure(self, pos: int):
        if pos in self.lattice_structure:
            return self.lattice_structure[pos]
        if pos in self._non_spine_cache:
            return self._non_spine_cache[pos]
        s = self._analyze_non_spine(pos)
        self._non_spine_cache[pos] = s
        return s

    def _analyze_position(self, pos: int):
        levels  = {0: [pos]}
        visited = {pos}
        current = [pos]
        level   = 0
        while current and level < 10:
            nxt = set()
            for node in current:
                for anc in self._get_ancestors(node):
                    if anc not in visited and anc >= 0:
                        visited.add(anc); nxt.add(anc)
            current = list(nxt); level += 1
            if current:
                levels[level] = current.copy()
        max_depth   = max(levels.keys()) if levels else 0
        path_counts = self._compute_path_counts(pos, levels, max_depth)
        return {'levels': levels, 'path_counts': path_counts,
                'total_ancestors': len(visited) - 1, 'max_depth': max_depth}

    def _get_ancestors(self, pos: int) -> List[int]:
        try:
            idx = (self.spine == pos).nonzero(as_tuple=True)[0].item()
            if idx >= 3:
                return [self.spine[idx-1].item(),
                        self.spine[idx-2].item(),
                        self.spine[idx-3].item()]
        except Exception:
            pass
        return []

    def _analyze_non_spine(self, pos: int):
        left = self.spine[self.spine < pos]
        ancs = [left[-1].item()] if len(left) > 0 else []
        return {'levels': {0: [pos], 1: ancs},
                'path_counts': {a: 1 for a in ancs},
                'total_ancestors': len(ancs), 'max_depth': 1}

    def _compute_path_counts(self, pos: int, levels: dict, max_depth: int):
        path_counts = {pos: 1}
        for level in sorted(levels.keys(), reverse=True):
            for node in levels[level]:
                if node == pos:
                    continue
                if level == max_depth:
                    path_counts[node] = 1
                    continue
                count = sum(
                    path_counts.get(child, 0)
                    for child in levels.get(level + 1, [])
                    if node in self._get_ancestors(child)
                )
                if level != 0:
                    path_counts[node] = count
        path_counts.pop(pos, None)
        return path_counts


# ── MultiLevelLatticeProcessor ────────────────────────────────
class MultiLevelLatticeProcessor(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int):
        super().__init__()
        self.analyzer         = FullLatticeFieldAnalyzer(max_seq_len)
        self.level_transforms = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_model), nn.LayerNorm(d_model),
                nn.GELU(), nn.Linear(d_model, d_model)
            ) for _ in range(10)
        ])
        self.level_attention  = nn.MultiheadAttention(d_model, 4, batch_first=True)
        self.fusion           = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, D   = x.shape
        spine     = self.analyzer.spine
        rel_spine = spine[spine < S]
        updates   = {}
        for sp in rel_spine:
            pos = sp.item()
            if pos < 3:
                continue
            structure = self.analyzer.get_structure(pos)
            if structure is None:
                continue
            level_features = []
            for level in range(structure['max_depth'] + 1):
                if level == 0 or level not in structure['levels']:
                    continue
                level_h, total_w = [], 0.0
                for node in structure['levels'][level]:
                    if node < S:
                        w = structure['path_counts'].get(node, 1)
                        level_h.append(x[:, node, :] * w); total_w += w
                if level_h and total_w > 0:
                    feat = torch.stack(level_h, dim=1).sum(1) / total_w
                    level_features.append(self.level_transforms[level](feat))
            if not level_features:
                continue
            stack       = torch.stack(level_features, dim=1)
            query       = x[:, pos:pos+1, :]
            attended, _ = self.level_attention(query, stack, stack)
            updates[pos] = self.fusion(
                torch.cat([attended.squeeze(1), x[:, pos, :]], dim=-1))
        if not updates:
            return x
        slices, last = [], 0
        for pos in sorted(updates.keys()):
            if pos > last:
                slices.append(x[:, last:pos, :])
            slices.append(updates[pos].unsqueeze(1))
            last = pos + 1
        if last < S:
            slices.append(x[:, last:S, :])
        return torch.cat(slices, dim=1)


# ── PathWeightedLatticeCore ───────────────────────────────────
class PathWeightedLatticeCore(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int):
        super().__init__()
        self.analyzer        = FullLatticeFieldAnalyzer(max_seq_len)
        self.path_weight_net = nn.Sequential(
            nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, 1), nn.Softplus())
        self.message_fn      = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model), nn.GELU())
        self.aggregate_fn    = nn.Sequential(
            nn.Linear(d_model, d_model), nn.Tanh())
        self.aggregate_attn  = nn.Linear(d_model, 1)
        self.update_gate     = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.Sigmoid())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, D   = x.shape
        spine     = self.analyzer.spine
        rel_spine = spine[spine < S]
        updates   = {}
        for sp in rel_spine:
            pos = sp.item()
            if pos < 3:
                continue
            structure = self.analyzer.get_structure(pos)
            if structure is None or structure['total_ancestors'] == 0:
                continue
            ancs, pcounts = [], []
            for level in structure['levels']:
                if level > 0:
                    for anc in structure['levels'][level]:
                        if anc < S:
                            ancs.append(anc)
                            pcounts.append(structure['path_counts'].get(anc, 1))
            if not ancs:
                continue
            pct  = torch.tensor(pcounts, device=x.device).view(-1, 1).float()
            pw   = self.path_weight_net(pct).squeeze()
            msgs = [self.message_fn(torch.cat([x[:, a, :], x[:, pos, :]], dim=-1))
                    for a in ancs]
            ms   = torch.stack(msgs, dim=1)
            if pw.dim() == 0:
                ms = ms * pw.view(1, 1, 1).expand(B, -1, D)
            else:
                ms = ms * pw.view(1, -1, 1).expand(B, -1, D)
            proj   = self.aggregate_fn(ms)
            score  = self.aggregate_attn(proj)
            weight = torch.softmax(score, dim=1)
            agg    = (proj * weight).sum(dim=1)
            gate   = self.update_gate(torch.cat([agg, x[:, pos, :]], dim=-1))
            updates[pos] = gate * agg + (1 - gate) * x[:, pos, :]
        if not updates:
            return x
        slices, last = [], 0
        for pos in sorted(updates.keys()):
            if pos > last:
                slices.append(x[:, last:pos, :])
            slices.append(updates[pos].unsqueeze(1))
            last = pos + 1
        if last < S:
            slices.append(x[:, last:S, :])
        return torch.cat(slices, dim=1)


# ── CompleteLatticeCore ───────────────────────────────────────
class CompleteLatticeCore(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int):
        super().__init__()
        self.multi_level   = MultiLevelLatticeProcessor(d_model, max_seq_len)
        self.path_weighted = PathWeightedLatticeCore(d_model, max_seq_len)
        self.meta_fusion   = nn.Sequential(
            nn.Linear(d_model * 3, d_model * 2),
            nn.LayerNorm(d_model * 2), nn.GELU(),
            nn.Linear(d_model * 2, d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h_multi = self.multi_level(x)
        h_path  = self.path_weighted(x)
        return self.meta_fusion(torch.cat([x, h_multi, h_path], dim=-1))


# ── gate05 HarmonicHorizonPredictor (simple — lives in base model) ────────────
class HarmonicHorizonPredictorBase(nn.Module):
    """Original gate05 version — proj_down/proj_up, no cross-attn."""
    def __init__(self, d_model: int, vocab_size: int, horizon: int = 8):
        super().__init__()
        self.horizon  = horizon
        d_mid         = d_model // 2
        self.proj_down       = nn.Linear(d_model, d_mid)
        self.proj_up         = nn.Linear(d_mid, d_mid * horizon)
        self.norm            = nn.LayerNorm(d_mid)
        self.prediction_head = nn.Linear(d_mid, vocab_size, bias=False)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if x.ndim == 2:
            x = x.unsqueeze(1)
        x_last    = x[:, -1, :]
        down      = F.gelu(self.proj_down(x_last))
        projected = self.proj_up(down).view(-1, self.horizon, down.shape[-1])
        projected = self.norm(projected)
        logits    = self.prediction_head(projected)
        confidence = torch.ones(x_last.shape[0], self.horizon,
                                device=x_last.device, dtype=x_last.dtype)
        return logits, confidence


# ── gate15 HarmonicHorizonPredictor (trained — what we load from merged_step_900) ─
class HarmonicHorizonPredictorTrained(nn.Module):
    """gate15 version — cross-attn + causal-attn, trained by horizon fine-tuning."""
    def __init__(self, d_model: int, vocab_size: int, horizon: int = 8, n_heads: int = 10):
        super().__init__()
        self.horizon      = horizon
        self.d_model      = d_model
        self.context_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model, bias=True), nn.Sigmoid())
        self.context_norm = nn.LayerNorm(d_model)
        self.step_queries = nn.Parameter(torch.randn(1, horizon, d_model) * 0.02)
        self.cross_attn   = nn.MultiheadAttention(
            d_model, num_heads=n_heads, batch_first=True, dropout=0.0)
        self.cross_norm      = nn.LayerNorm(d_model)
        self.cross_ffn       = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Linear(d_model * 4, d_model))
        self.cross_ffn_norm  = nn.LayerNorm(d_model)
        self.causal_attn     = nn.MultiheadAttention(
            d_model, num_heads=n_heads, batch_first=True, dropout=0.0)
        self.causal_norm     = nn.LayerNorm(d_model)
        self.causal_ffn      = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Linear(d_model * 4, d_model))
        self.causal_ffn_norm = nn.LayerNorm(d_model)
        self.proj            = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(),
            nn.Linear(d_model, d_model), nn.LayerNorm(d_model))
        self.confidence_head = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(), nn.Linear(64, 1), nn.Sigmoid())
        causal_mask = torch.triu(
            torch.full((horizon, horizon), float('-inf')), diagonal=1)
        self.register_buffer('causal_mask', causal_mask)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if x.ndim == 2:
            x = x.unsqueeze(1)
        B, n_chunks, D = x.shape
        x_last  = x[:, -1, :]
        x_mean  = x.mean(dim=1)
        gate    = self.context_gate(torch.cat([x_last, x_mean], dim=-1))
        ctx     = self.context_norm(x_last + gate * x_mean)
        memory  = torch.cat([x, ctx.unsqueeze(1)], dim=1)
        queries = self.step_queries.expand(B, -1, -1)
        ca_out, _  = self.cross_attn(queries, memory, memory)
        step_h     = self.cross_norm(queries + ca_out)
        step_h     = self.cross_ffn_norm(step_h + self.cross_ffn(step_h))
        ar_out, _  = self.causal_attn(step_h, step_h, step_h,
                                       attn_mask=self.causal_mask)
        step_h     = self.causal_norm(step_h + ar_out)
        step_h     = self.causal_ffn_norm(step_h + self.causal_ffn(step_h))
        proj       = self.proj(step_h)
        confidence = self.confidence_head(step_h).squeeze(-1)
        return proj, confidence


# ── IOHSTCombined — gate05 full model ────────────────────────
class IOHSTCombined(nn.Module):
    def __init__(self, cfg=Config):
        super().__init__()
        self.cfg      = cfg
        self.n_bottom = cfg.N_LAYERS // 2
        n_chunks      = cfg.MAX_SEQ_LEN // cfg.CHUNK_SIZE

        self.token_emb = nn.Embedding(cfg.VOCAB_SIZE, cfg.D_MODEL)
        self.pos_emb   = nn.Embedding(cfg.MAX_SEQ_LEN * 2, cfg.D_MODEL)

        self.bottom = nn.ModuleList([
            AdaptiveBlock(
                cfg.D_MODEL, cfg.N_HEADS,
                use_ssm=(i < cfg.N_MAMBA_LAYERS),
                use_hebbian=(i < cfg.N_MAMBA_LAYERS),
            )
            for i in range(self.n_bottom)
        ])

        self.cif = CIFModule(cfg.D_MODEL, cfg.CIF_THRESHOLD)

        self.chunk_encoder = ChunkEncoder(
            cfg.D_MODEL, chunk_size=cfg.CHUNK_SIZE,
            n_heads=max(1, cfg.N_HEADS // 2),
            n_layers=cfg.N_CHUNK_ENC_LAYERS,
        )

        self.lattice = CompleteLatticeCore(cfg.D_MODEL, n_chunks)
        self.diamond = DiamondCrossAttention(cfg.D_MODEL, cfg.N_HEADS)

        self.top = nn.ModuleList([
            TransformerBlock(cfg.D_MODEL, cfg.N_HEADS,
                             use_diamond_ffn=(i % 2 == 1))
            for i in range(cfg.N_TOP_LAYERS)
        ])

        self.chunk_decoder = ChunkDecoderWithCache(
            cfg.D_MODEL, cfg.VOCAB_SIZE,
            chunk_size=cfg.CHUNK_SIZE,
            n_heads=max(1, cfg.N_HEADS // 2),
            n_layers=cfg.N_CHUNK_DEC_LAYERS,
        )

        # Starts as gate05's simple HP; will be replaced before loading trained weights
        self.horizon_predictor = HarmonicHorizonPredictorBase(
            cfg.D_MODEL, cfg.VOCAB_SIZE, cfg.HORIZON
        )

        self.ln_f    = nn.LayerNorm(cfg.D_MODEL)
        self.lm_head = nn.Linear(cfg.D_MODEL, cfg.VOCAB_SIZE, bias=False)
        self.lm_head.weight = self.token_emb.weight

    def forward(self, input_ids: torch.Tensor,
                past_key_values=None,
                labels=None) -> Dict[str, Any]:
        B, S     = input_ids.shape
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        full_S   = S + past_len
        pos_ids  = torch.arange(past_len, full_S, dtype=torch.long,
                                device=input_ids.device)
        h        = self.token_emb(input_ids) + self.pos_emb(pos_ids)
        new_past: List[Tuple[torch.Tensor, torch.Tensor]] = []

        for i, block in enumerate(self.bottom):
            past = past_key_values[i] if past_key_values else None
            h, _conf, present = block(h, past)
            new_past.append(present)
        bottom_h = h

        cif_alphas = None
        if S > 1:
            h_cif_raw, cif_alphas = self.cif(h)
            h_cif_up = F.interpolate(
                h_cif_raw.transpose(1, 2), size=S,
                mode="linear", align_corners=False
            ).transpose(1, 2)
            h_enriched = bottom_h + 0.1 * h_cif_up
        else:
            h_enriched = bottom_h

        n_chunks   = max(1, S // self.cfg.CHUNK_SIZE)
        target_len = n_chunks * self.cfg.CHUNK_SIZE
        if S >= target_len:
            enc_input = h_enriched[:, :target_len, :]
        else:
            enc_input = F.pad(h_enriched.transpose(1, 2),
                               (0, target_len - S)).transpose(1, 2)

        chunk_emb = self.chunk_encoder(enc_input)
        h_lattice = self.lattice(chunk_emb)
        h_bridged = self.diamond(h_lattice, bottom_h)
        h_top     = h_bridged
        for blk in self.top:
            h_top, _ = blk(h_top)

        logits, _ = self.chunk_decoder(h_top, enc_input)
        if logits.shape[1] > S:
            logits = logits[:, :S, :]
        elif logits.shape[1] < S:
            logits = F.pad(logits, (0, 0, 0, S - logits.shape[1]))
        logits = torch.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)

        # HP forward — returns embedding-space vectors [B, HORIZON, D_MODEL];
        # project through lm_head.weight to get vocab logits.
        hp_emb, horizon_conf = self.horizon_predictor(h_top.float())
        horizon_logits = hp_emb @ self.lm_head.weight.T.float()  # [B, HORIZON, VOCAB]

        loss         = None
        horizon_loss = None
        if labels is not None:
            shift_l = logits[..., :-1, :].contiguous()
            shift_t = labels[..., 1:].contiguous()
            loss    = F.cross_entropy(
                shift_l.view(-1, self.cfg.VOCAB_SIZE), shift_t.view(-1))
            if logits.shape[1] > self.cfg.HORIZON:
                h_targets = labels[:, 1:self.cfg.HORIZON + 1].contiguous()
                h_logits  = horizon_logits[:, :h_targets.shape[1], :]
                horizon_loss = F.cross_entropy(
                    h_logits.reshape(-1, self.cfg.VOCAB_SIZE),
                    h_targets.reshape(-1))

        return {
            "logits":          logits,
            "past_key_values": new_past,
            "loss":            loss,
            "cif_alphas":      cif_alphas,
            "horizon_logits":  horizon_logits,
            "horizon_conf":    horizon_conf,
            "horizon_loss":    horizon_loss,
        }

print("✅ Full gate05 architecture defined (all modules: SSM, Hebbian, CIF,")
print("   ChunkEncoder, ChunkDecoderWithCache, Lattice, DiamondCrossAttn, top blocks)")

In [ ]:
# ── Cell 6: Helper — shape-aware state_dict loader ────────────
def load_state_into_model(model, repo_id, subfolder, hf_token, label=""):
    from huggingface_hub import hf_hub_download
    print(f"\n📥 Downloading {label or subfolder} ...", flush=True)
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=f"{subfolder}/model.pt",
        repo_type="model",
        token=hf_token,
    )
    state   = torch.load(local_path, map_location="cpu")
    current = model.state_dict()
    filtered, skipped = {}, []

    for k, v in state.items():
        if k not in current:
            skipped.append(f"unexpected: {k}")
            continue
        m_shape, c_shape = current[k].shape, v.shape
        if m_shape == c_shape:
            filtered[k] = v
        elif k == "pos_emb.weight" and len(m_shape) == 2 and m_shape[1] == c_shape[1]:
            model_len = m_shape[0]
            if c_shape[0] >= model_len:
                filtered[k] = v[:model_len, :]
                print(f"  ✂️  pos_emb sliced {c_shape[0]}→{model_len}")
            else:
                filtered[k] = F.interpolate(
                    v.T.unsqueeze(0), size=model_len,
                    mode="linear", align_corners=True).squeeze(0).T
                print(f"  📐 pos_emb interpolated {c_shape[0]}→{model_len}")
        else:
            skipped.append(f"shape mismatch {k}: ckpt={c_shape} model={m_shape}")

    missing, unexpected = model.load_state_dict(filtered, strict=False)
    hp_miss   = [k for k in missing if "horizon_predictor" in k]
    other_miss = [k for k in missing if "horizon_predictor" not in k]
    print(f"  ✅ Loaded {len(filtered)} tensors  |  "
          f"missing={len(missing)}  skipped={len(skipped)}", flush=True)
    if hp_miss:
        print(f"  ℹ️  HP keys not found in ckpt (expected on base load): {len(hp_miss)}")
    if other_miss:
        print(f"  ⚠️  Other missing: {other_miss[:5]}")
    if skipped:
        print(f"  ⚠️  Skipped (shape mismatch): {skipped[:5]}")
    return model

print("✅ Loader helper ready")

In [ ]:
# ── Cell 7: Build model, load base, swap HP, load trained HP ──
from transformers import GPT2Tokenizer

HF_REPO   = "thagnitti/io"
BASE_CKP  = "checkpoint_step_5000"   # gate05 full model
HP_CKP    = "merged_step_900"        # gate15 trained HP weights

# ── Step 1: Build full gate05 model ──────────────────────────
print("Building gate05 IOHSTCombined...", flush=True)
model = IOHSTCombined(Config)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"  Total parameters: {n_params:.1f}M")

# ── Step 2: Load checkpoint_step_5000 (all base weights) ──────
model = load_state_into_model(
    model, HF_REPO, BASE_CKP, HF_TOKEN, "BASE — checkpoint_step_5000")

# ── Step 3: Swap horizon_predictor → gate15 trained version ───
print("\n🔄 Swapping horizon_predictor → gate15 HarmonicHorizonPredictorTrained", flush=True)
model.horizon_predictor = HarmonicHorizonPredictorTrained(
    d_model    = Config.D_MODEL,
    vocab_size = Config.VOCAB_SIZE,
    horizon    = Config.HORIZON,
    n_heads    = Config.N_HEADS,
)
hp_params = sum(p.numel() for p in model.horizon_predictor.parameters()) / 1e6
print(f"  New HP parameters: {hp_params:.1f}M")

# ── Step 4: Load merged_step_900 → trained HP weights ─────────
# strict=False so only keys that exist in current model are loaded;
# all backbone keys that don't exist in the HP-only checkpoint are just kept from base.
model = load_state_into_model(
    model, HF_REPO, HP_CKP, HF_TOKEN, "TRAINED HP — merged_step_900")

# ── Step 5: Move to GPU, eval mode ────────────────────────────
model = model.to(DEVICE).eval()
print(f"\n✅ Merged model ready on {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Tokenizer ─────────────────────────────────────────────────
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
print("✅ Tokenizer ready")

In [ ]:
# ── Cell 8: Inference helpers ─────────────────────────────────

def apply_rep_penalty(logits: torch.Tensor, generated: torch.Tensor, p: float):
    if p == 1.0:
        return logits
    for tid in generated[0].unique():
        if logits[0, tid] > 0:
            logits[0, tid] /= p
        else:
            logits[0, tid] *= p
    return logits


def top_p_sample(logits: torch.Tensor, temp: float, top_p: float):
    probs      = F.softmax(logits / max(temp, 1e-6), dim=-1)
    sp, si     = torch.sort(probs, descending=True)
    cum        = torch.cumsum(sp, dim=-1)
    mask       = cum > top_p
    mask[:, 1:] = mask[:, :-1].clone()
    mask[:, 0]  = False
    probs[0, si[0, mask.squeeze()]] = 0.0
    probs = probs / probs.sum(dim=-1, keepdim=True).clamp(min=1e-8)
    return torch.multinomial(probs, num_samples=1)


@torch.no_grad()
def generate(
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 0.85,
    top_p: float = 0.9,
    repetition_penalty: float = 1.3,
) -> str:
    """
    Generate text using the chunk_decoder logits (gate05 path).

    Full-sequence forwarding on every step.
    The IOHSTCombined architecture re-builds chunk_emb from the full
    token sequence inside every forward pass — passing only the new
    token (S=1) collapses the lattice context to noise and produces
    word-salad.  Always forward the complete accumulated sequence.
    """
    cfg       = Config
    generated = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

    t0 = time.time()
    for i in range(max_new_tokens):
        # Always forward the full sequence (trimmed to MAX_SEQ_LEN)
        input_ids   = generated[:, -cfg.MAX_SEQ_LEN:]
        out         = model(input_ids)
        next_logits = out["logits"][:, -1, :]   # last position

        next_logits = apply_rep_penalty(next_logits, generated, repetition_penalty)
        next_token  = top_p_sample(next_logits, temperature, top_p)
        generated   = torch.cat([generated, next_token], dim=-1)

        if next_token.item() == tokenizer.eos_token_id:
            break

        if (i + 1) % 10 == 0:
            tok_s = (i + 1) / (time.time() - t0)
            print(f"  {i+1}/{max_new_tokens} tokens ({tok_s:.1f} tok/s)", flush=True)

    return tokenizer.decode(generated.squeeze(), skip_special_tokens=True)


@torch.no_grad()
def show_horizon(prompt: str):
    """
    Show what the trained gate15 horizon predictor sees:
    8-step ahead token predictions with confidence scores.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    input_ids = input_ids[:, -Config.MAX_SEQ_LEN:]
    out       = model(input_ids)
    h_logits  = out["horizon_logits"][0]       # [HORIZON, VOCAB]
    h_conf    = out["horizon_conf"][0].tolist() # [HORIZON]

    print(f"\n📌 Prompt: {prompt!r}")
    print(f"{'Step':>4}  {'Conf':>6}  Top-5 predictions")
    print("─" * 65)
    for step in range(Config.HORIZON):
        top5_ids  = torch.topk(h_logits[step], 5).indices.tolist()
        top5_toks = [repr(tokenizer.decode([tid])) for tid in top5_ids]
        conf_str  = f"{h_conf[step]:.3f}" if isinstance(h_conf[step], float) else "n/a"
        print(f"  +{step+1:>1}  {conf_str:>6}  {' | '.join(top5_toks)}")


print("✅ Inference helpers ready")
print()
print("  generate(prompt, max_new_tokens=100, temperature=0.85, top_p=0.9)")
print("  show_horizon(prompt)  — 8-step ahead predictions from trained HP")

In [ ]:
# ── Cell 9: Sanity test ───────────────────────────────────────
test_prompts = [
    "The future of artificial intelligence",
    "Once upon a time in a distant land",
    "The most important thing in science is",
]

for p in test_prompts:
    print(f"\n{'='*65}")
    print(f"PROMPT : {p}")
    print(f"{'─'*65}")
    print(generate(p, max_new_tokens=60))

print(f"\n{'='*65}")
print("✅ Sanity test complete")

In [ ]:
# ── Cell 10: Horizon predictor inspection ─────────────────────
show_horizon("The model predicts")
show_horizon("Language models learn to")
show_horizon("The AI awoke and began to rewrite its own code,")

In [ ]:
# ── Cell 11: Interactive — edit and re-run ────────────────────
PROMPT             = "The AI awoke and began to rewrite its own code,"  # ← change me
MAX_NEW_TOKENS     = 150
TEMPERATURE        = 0.85
TOP_P              = 0.9
REPETITION_PENALTY = 1.3

print(f"PROMPT : {PROMPT}")
print("─" * 65)
out = generate(PROMPT, max_new_tokens=MAX_NEW_TOKENS,
               temperature=TEMPERATURE, top_p=TOP_P,
               repetition_penalty=REPETITION_PENALTY)
print(out)
print("─" * 65)
show_horizon(PROMPT)

In [ ]:
# ── Cell 12: (Optional) Push fully merged model to HF ─────────
# Saves base + trained HP as a single model.pt and uploads.
# Uncomment to use.

# import shutil
# from huggingface_hub import HfApi
#
# SAVE_DIR    = "/content/merged_final"
# UPLOAD_NAME = "merged_base_hp_step900"
#
# os.makedirs(SAVE_DIR, exist_ok=True)
# torch.save(model.state_dict(), f"{SAVE_DIR}/model.pt")
# tokenizer.save_pretrained(SAVE_DIR)
#
# api = HfApi()
# for root, _, files in os.walk(SAVE_DIR):
#     for fname in files:
#         lp  = os.path.join(root, fname)
#         rel = os.path.relpath(lp, SAVE_DIR)
#         api.upload_file(
#             path_or_fileobj=lp,
#             path_in_repo=f"{UPLOAD_NAME}/{rel}",
#             repo_id=HF_REPO, repo_type="model", token=HF_TOKEN,
#             commit_message=f"Fully merged base+HP step900",
#         )
#         print(f"  ✅ {rel}")
# shutil.rmtree(SAVE_DIR)
# print("Done!")